# RL Vectorizer — Diagnostic Experiments

两个诊断实验，定位模型学不会 SVG 的瓶颈。

In [ ]:
# Cell 1: Clone + install
!git clone https://github.com/dersteppenwolfruowen-316/RL-Vec.git
%cd /content/RL-Vec

!pip install -q transformers peft accelerate bitsandbytes datasets
!pip install -q lxml cairosvg pillow scikit-image opencv-python tensorboard
!pip install -q hydra-core omegaconf shapely
!pip uninstall -y torchao

import torch
print(f"CUDA: {torch.cuda.is_available()}")
print('OK')

In [ ]:
# Cell 2: Download ResPlan + convert
import urllib.request, zipfile, os

DATA_DIR = "data/resplan"
os.makedirs(DATA_DIR, exist_ok=True)
ZIP = os.path.join(DATA_DIR, "ResPlan.zip")

if not os.path.exists(ZIP):
    print("Downloading ResPlan.zip (96MB)...")
    urllib.request.urlretrieve(
        "https://github.com/m-agour/ResPlan/releases/download/1.0.0/ResPlan.zip", ZIP)

if not os.path.exists(os.path.join(DATA_DIR, "ResPlan.pkl")):
    print("Extracting...")
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(DATA_DIR)

if not os.path.exists(os.path.join(DATA_DIR, "svgs")):
    !python convert_resplan.py

if not os.path.exists(os.path.join(DATA_DIR, "sft_train.jsonl")):
    !python scripts/prepare_sft_data.py

print('OK')

---
## 实验 1：房间计数 + 位置感知

50 条数据，输出结构化摘要（房间数 + 位置），验证 VLM 能否从 bitmap 感知房间空间布局。

In [ ]:
# Cell 3: Experiment 1 — Room Count
!python scripts/train_diag_roomcount.py --max-samples 50 --epochs 3

---
## 实验 2：离散网格坐标 (256×256)

200 条数据，SVG 坐标从连续浮点数 → 0-255 整数网格。验证坐标表示是否是瓶颈。

In [ ]:
# Cell 4: Experiment 2 — Grid SVG
!python scripts/train_diag_grid.py --max-samples 200 --epochs 3